# Dynamic Spine Chain Prototype

This notebook is a proof-of-idea for post-processing YOLO pose vertebra candidates into one anatomically plausible spine chain.

Key idea: we do not provide the number of vertebrae. YOLO predicts many candidates at a low confidence threshold, then the chain algorithm chooses a variable-length ordered path. A candidate is kept only when its confidence and geometry improve the chain enough.

In [ ]:
from pathlib import Path
import math
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    from ultralytics import YOLO
except ImportError as exc:
    raise ImportError("Install ultralytics first: pip install ultralytics") from exc

plt.rcParams["figure.figsize"] = (10, 14)

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "weights" / "best.pt").exists() and (path / "dataset" / "processed" / "yolo").exists():
            return path
    raise FileNotFoundError("Could not find the Spine-Opportunistic-Screening project root.")


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / "src" / "weights" / "best.pt"
YOLO_ROOT = PROJECT_ROOT / "dataset" / "processed" / "yolo"

# Use AJ images for inference-only testing. Switch SOURCE_NAME to "yolo_test" if you want GT count checks.
SOURCE_NAME = "from_AJ"
if SOURCE_NAME == "from_AJ":
    IMAGE_DIR = PROJECT_ROOT / "dataset" / "from_AJ"
    LABEL_DIR = None
elif SOURCE_NAME == "yolo_test":
    IMAGE_SPLIT = "test"
    IMAGE_DIR = YOLO_ROOT / "images" / IMAGE_SPLIT
    LABEL_DIR = YOLO_ROOT / "labels" / IMAGE_SPLIT
else:
    raise ValueError(f"Unsupported SOURCE_NAME: {SOURCE_NAME}")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "spine_chain_prototype" / SOURCE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_PATH)
print("Source:", SOURCE_NAME)
print("Images:", IMAGE_DIR)
print("Labels:", LABEL_DIR if LABEL_DIR is not None else "none")
print("Output:", OUTPUT_DIR)

## Why the chain can work without knowing the count

The algorithm treats vertebra count as an output, not an input:

1. Run YOLO with low `conf` to collect enough possible vertebrae.
2. Remove obvious duplicate detections.
3. Estimate typical vertebra spacing from candidate sizes and vertical gaps.
4. Build a directed graph from superior to inferior candidates.
5. Find the best scoring path. Each candidate adds reward only if its confidence is above a soft acceptance threshold. Bad spacing, lateral jumps, and size jumps subtract score.
6. The path stops naturally when adding another candidate makes the total score worse.

In [ ]:
def list_images(folder: Path):
    valid_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    return sorted(path for path in folder.iterdir() if path.is_file() and path.suffix.lower() in valid_suffixes)


# Pick an image. Change IMAGE_INDEX to inspect other cases.
image_paths = list_images(IMAGE_DIR)
assert image_paths, f"No images found in {IMAGE_DIR}"

IMAGE_INDEX = 0
# IMAGE_INDEX = random.randrange(len(image_paths))
image_path = image_paths[IMAGE_INDEX]
label_path = LABEL_DIR / f"{image_path.stem}.txt" if LABEL_DIR is not None else None

print("Image:", image_path.name)
print("Label exists:", bool(label_path and label_path.exists()))

In [ ]:
model = YOLO(str(MODEL_PATH))

# Use low conf for candidate generation. The spine-chain will decide which detections survive.
PRED_CONF = 0.05
PRED_IOU = 0.60
MAX_DET = 80

result = model.predict(
    source=str(image_path),
    imgsz=1024,
    conf=PRED_CONF,
    iou=PRED_IOU,
    max_det=MAX_DET,
    verbose=False,
)[0]

image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_h, image_w = image_rgb.shape[:2]
print("Image shape:", image_w, image_h)

In [ ]:
def yolo_result_to_candidates(result):
    if result.boxes is None or result.keypoints is None:
        return []

    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    keypoints = result.keypoints.xy.detach().cpu().numpy().astype(np.float32)

    candidates = []
    for idx, (box, score, kpts) in enumerate(zip(boxes, scores, keypoints)):
        x1, y1, x2, y2 = box.tolist()
        valid = np.isfinite(kpts).all(axis=1) & (kpts[:, 0] > 0) & (kpts[:, 1] > 0)
        if valid.any():
            center = kpts[valid].mean(axis=0)
        else:
            center = np.array([(x1 + x2) * 0.5, (y1 + y2) * 0.5], dtype=np.float32)

        candidates.append({
            "idx": int(idx),
            "score": float(score),
            "box": np.array([x1, y1, x2, y2], dtype=np.float32),
            "kpts": kpts,
            "x": float(center[0]),
            "y": float(center[1]),
            "w": float(max(x2 - x1, 1.0)),
            "h": float(max(y2 - y1, 1.0)),
        })
    return candidates


raw_candidates = yolo_result_to_candidates(result)
print("Raw candidates:", len(raw_candidates))

In [ ]:
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union


def suppress_duplicate_candidates(candidates, iou_threshold=0.18, center_scale=0.35):
    kept = []
    for cand in sorted(candidates, key=lambda c: -c["score"]):
        duplicate = False
        for prev in kept:
            iou = box_iou(cand["box"], prev["box"])
            dist = math.hypot(cand["x"] - prev["x"], cand["y"] - prev["y"])
            radius = center_scale * min(cand["h"], prev["h"])
            if iou >= iou_threshold or dist <= radius:
                duplicate = True
                break
        if not duplicate:
            kept.append(cand)
    return sorted(kept, key=lambda c: (c["y"], c["x"]))


dedup_candidates = suppress_duplicate_candidates(raw_candidates)
print("After duplicate suppression:", len(dedup_candidates))

In [ ]:
def estimate_spacing(candidates):
    if not candidates:
        return 32.0, 16.0, 80.0

    heights = np.array([c["h"] for c in candidates], dtype=np.float32)
    median_h = float(np.median(heights)) if len(heights) else 32.0
    ys = np.array(sorted(c["y"] for c in candidates), dtype=np.float32)
    gaps = np.diff(ys)
    plausible = gaps[(gaps >= 0.35 * median_h) & (gaps <= 2.20 * median_h)]

    if len(plausible) >= 2:
        target_dy = float(np.median(plausible))
    else:
        target_dy = max(0.85 * median_h, 12.0)

    min_dy = max(0.35 * target_dy, 8.0)
    max_dy = max(2.10 * target_dy, min_dy + 8.0)
    return target_dy, min_dy, max_dy


target_dy, min_dy, max_dy = estimate_spacing(dedup_candidates)
print({"target_dy": round(target_dy, 2), "min_dy": round(min_dy, 2), "max_dy": round(max_dy, 2)})

In [ ]:
def transition_score(prev, curr, target_dy, min_dy, max_dy):
    dy = curr["y"] - prev["y"]
    if dy <= 0 or dy < min_dy or dy > max_dy:
        return None

    dx = curr["x"] - prev["x"]
    if abs(dx) > 1.35 * max(target_dy, 1.0):
        return None

    spacing_penalty = ((dy - target_dy) / max(target_dy, 1.0)) ** 2
    lateral_penalty = (dx / max(target_dy, 1.0)) ** 2
    size_penalty = abs(math.log(max(curr["h"], 1.0) / max(prev["h"], 1.0)))

    return -(0.55 * spacing_penalty + 0.35 * lateral_penalty + 0.20 * size_penalty)


def dynamic_spine_chain(
    candidates,
    score_threshold=0.18,
    score_weight=3.0,
    min_chain_len=3,
):
    ordered = sorted(candidates, key=lambda c: (c["y"], c["x"]))
    if not ordered:
        return [], {}

    target_dy, min_dy, max_dy = estimate_spacing(ordered)
    n = len(ordered)
    best = np.full(n, -np.inf, dtype=np.float64)
    length = np.ones(n, dtype=np.int32)
    parent = np.full(n, -1, dtype=np.int32)

    node_values = np.array([score_weight * (c["score"] - score_threshold) for c in ordered], dtype=np.float64)
    best[:] = node_values

    for i in range(n):
        for j in range(i):
            edge = transition_score(ordered[j], ordered[i], target_dy, min_dy, max_dy)
            if edge is None:
                continue
            value = best[j] + node_values[i] + edge
            if value > best[i]:
                best[i] = value
                length[i] = length[j] + 1
                parent[i] = j

    valid_ends = np.where(length >= min_chain_len)[0]
    if len(valid_ends) == 0:
        valid_ends = np.arange(n)

    end_idx = int(valid_ends[np.argmax(best[valid_ends])])
    chain_indices = []
    cur = end_idx
    while cur >= 0:
        chain_indices.append(cur)
        cur = int(parent[cur])
    chain_indices.reverse()

    chain = [ordered[i] for i in chain_indices]
    debug = {
        "score": float(best[end_idx]),
        "target_dy": float(target_dy),
        "min_dy": float(min_dy),
        "max_dy": float(max_dy),
        "node_score_threshold": float(score_threshold),
        "num_candidates": int(n),
        "num_selected": int(len(chain)),
    }
    return chain, debug


chain, chain_debug = dynamic_spine_chain(dedup_candidates)
print(chain_debug)

In [ ]:
def read_yolo_pose_label(label_path, image_w, image_h):
    if label_path is None or not label_path.exists():
        return []

    rows = []
    for line in label_path.read_text().splitlines():
        parts = [float(x) for x in line.split()]
        if len(parts) < 5 + 4 * 3:
            continue
        cls, cx, cy, bw, bh = parts[:5]
        raw_kpts = np.array(parts[5:], dtype=np.float32).reshape(4, 3)
        kpts = raw_kpts[:, :2].copy()
        kpts[:, 0] *= image_w
        kpts[:, 1] *= image_h
        center = kpts.mean(axis=0)
        rows.append({
            "center": center,
            "kpts": kpts,
            "box": np.array([
                (cx - bw / 2.0) * image_w,
                (cy - bh / 2.0) * image_h,
                (cx + bw / 2.0) * image_w,
                (cy + bh / 2.0) * image_h,
            ], dtype=np.float32),
        })
    return sorted(rows, key=lambda r: (float(r["center"][1]), float(r["center"][0])))


gt_rows = read_yolo_pose_label(label_path, image_w, image_h)
print("GT vertebrae:", len(gt_rows))
print("Selected chain:", len(chain))
if gt_rows:
    print("Count error:", len(chain) - len(gt_rows))

In [ ]:
def draw_box(ax, box, color, linewidth=1.2, alpha=1.0):
    x1, y1, x2, y2 = box
    ax.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], color=color, linewidth=linewidth, alpha=alpha)


def draw_poly(ax, kpts, color, linewidth=1.8, alpha=1.0):
    order = [0, 1, 3, 2, 0]
    pts = kpts[order]
    ax.plot(pts[:, 0], pts[:, 1], color=color, linewidth=linewidth, alpha=alpha)
    ax.scatter(kpts[:, 0], kpts[:, 1], s=22, c=color, alpha=alpha)


def visualize_chain(image_rgb, candidates, chain, gt_rows=None, title=""):
    fig, ax = plt.subplots(figsize=(10, 14))
    ax.imshow(image_rgb, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

    chain_ids = {id(c) for c in chain}
    for cand in candidates:
        if id(cand) in chain_ids:
            continue
        draw_box(ax, cand["box"], color="dodgerblue", linewidth=0.8, alpha=0.35)
        ax.scatter([cand["x"]], [cand["y"]], s=12, c="dodgerblue", alpha=0.45)

    if gt_rows:
        for gt in gt_rows:
            draw_poly(ax, gt["kpts"], color="white", linewidth=1.0, alpha=0.55)

    if chain:
        centers = np.array([[c["x"], c["y"]] for c in chain], dtype=np.float32)
        ax.plot(centers[:, 0], centers[:, 1], color="lime", linewidth=2.0, alpha=0.95)
        for rank, cand in enumerate(chain, start=1):
            draw_poly(ax, cand["kpts"], color="lime", linewidth=2.2, alpha=0.95)
            ax.text(
                cand["x"], cand["y"], f"{rank}:{cand['score']:.2f}",
                color="black", fontsize=9, ha="center", va="center",
                bbox={"boxstyle": "round,pad=0.2", "fc": "lime", "ec": "none", "alpha": 0.8},
            )

    return fig, ax


title = f"{image_path.name} | raw={len(raw_candidates)} dedup={len(dedup_candidates)} chain={len(chain)} gt={len(gt_rows)}"
fig, ax = visualize_chain(image_rgb, dedup_candidates, chain, gt_rows=gt_rows, title=title)
preview_path = OUTPUT_DIR / f"{image_path.stem}_spine_chain.png"
fig.savefig(preview_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", preview_path)

## Parameters to tune first

- `PRED_CONF`: lower means more candidates and fewer misses, but more false positives.
- `score_threshold` in `dynamic_spine_chain`: higher means shorter, stricter chains.
- `min_dy` and `max_dy` logic in `estimate_spacing`: controls whether the chain can bridge a missed vertebra.
- lateral jump limit in `transition_score`: controls false positives outside the spine line.

A good next experiment is to loop over 20 to 50 validation/test images and inspect count error, missed vertebrae, and false positives.

In [ ]:
def run_one_image(path, save_preview=True):
    r = model.predict(source=str(path), imgsz=1024, conf=PRED_CONF, iou=PRED_IOU, max_det=MAX_DET, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(str(path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    raw = yolo_result_to_candidates(r)
    dedup = suppress_duplicate_candidates(raw)
    selected, debug = dynamic_spine_chain(dedup)
    label = LABEL_DIR / f"{path.stem}.txt" if LABEL_DIR is not None else None
    gt = read_yolo_pose_label(label, w, h)

    if save_preview:
        title = f"{path.name} | raw={len(raw)} dedup={len(dedup)} chain={len(selected)} gt={len(gt)}"
        fig, _ = visualize_chain(img, dedup, selected, gt_rows=gt, title=title)
        fig.savefig(OUTPUT_DIR / f"{path.stem}_spine_chain.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    return {
        "image": path.name,
        "raw": len(raw),
        "dedup": len(dedup),
        "chain": len(selected),
        "gt": len(gt),
        "count_error": len(selected) - len(gt) if gt else None,
        "score": debug.get("score", None),
    }


sample_paths = image_paths
rows = [run_one_image(path, save_preview=True) for path in sample_paths]
print(f"Saved {len(rows)} previews to {OUTPUT_DIR}")
rows[:5]